In [3]:
import sys, os, traceback
sys.path.insert(0, os.path.abspath('..'))

import importlib
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

import data_pipeline.config_variables as _cv;  importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import data_pipeline.config_cluster as _cc;  importlib.reload(_cc)
from data_pipeline.config_cluster import (
    N_CLUSTERS_LOCAL, TEST_MODE, TEST_LA_CODES, WAVE, HIERARCHICAL_CLUSTER
)

import data_pipeline.helpers.cluster_summary as _cs;  importlib.reload(_cs)
import data_pipeline.helpers.llm_prompts as _lp;  importlib.reload(_lp)
from data_pipeline.helpers.llm_prompts import resolve_emp_label
import data_pipeline.helpers.llm_cluster_pipeline as _lcp;  importlib.reload(_lcp)
import data_pipeline.helpers.llm_claude as _lclaude;  importlib.reload(_lclaude)

# ── Config (edit these) ────────────────────────────────────────────────────────────────────────────
MAX_PIDPS_PER_GROUP = 100       # top N PIDPs (by weight) per LA × group
MODEL               = "claude-opus-4-5"   # change to any claude-* you have access to
THINKING_BUDGET     = 8000      # set to 0 to disable extended thinking
RETRY_DELAY         = 10
hier_col = f"{WAVE}_{HIERARCHICAL_CLUSTER}" if HIERARCHICAL_CLUSTER else None

print(f"WAVE={WAVE}  HIERARCHICAL_CLUSTER={HIERARCHICAL_CLUSTER}  MODEL={MODEL}")
print(f"MAX_PIDPS_PER_GROUP={MAX_PIDPS_PER_GROUP}  THINKING_BUDGET={THINKING_BUDGET}")

# ── Paths ─────────────────────────────────────────────────────────────────────────────────
NL_PROFILE = Path(f"../{DATA_FOLDER}/5_add_nl_strings/{WAVE}_with_nl_profile.pkl")
LA_COUNTS  = Path(f"../{DATA_FOLDER}/10_synthetic_population/pidp_la_counts.csv")
API_OUT    = Path("../api/data/clusters/local_llm_claude_clusters.csv")
PIDP_OUT   = Path(f"../{DATA_FOLDER}/15_llm_cluster_LA_claude/claude_pidp_assignments.csv")

for p in (NL_PROFILE, LA_COUNTS):
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")
API_OUT.parent.mkdir(parents=True, exist_ok=True)
PIDP_OUT.parent.mkdir(parents=True, exist_ok=True)

if API_OUT.exists() != PIDP_OUT.exists():
    for f in (API_OUT, PIDP_OUT):
        if f.exists():
            f.unlink()
            print(f"Removed {f.name} to resync outputs.")

# ── API key & caller ──────────────────────────────────────────────────────────────────────
load_dotenv(Path("..") / ".env")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
if not ANTHROPIC_API_KEY:
    raise EnvironmentError("ANTHROPIC_API_KEY not set. Add to .env or export in shell.")
print(f"ANTHROPIC_API_KEY loaded (length={len(ANTHROPIC_API_KEY)}, prefix={ANTHROPIC_API_KEY[:6]}…)", flush=True)

_llm_chat = _lclaude.make_claude_caller(ANTHROPIC_API_KEY, MODEL, THINKING_BUDGET)
print(f"Claude client ready ({MODEL}" +
      (f", extended thinking budget={THINKING_BUDGET}" if THINKING_BUDGET > 0 else "") +
      f").  PIDP → {PIDP_OUT}")


WAVE=k  HIERARCHICAL_CLUSTER=jbstat_eng  MODEL=claude-opus-4-5
MAX_PIDPS_PER_GROUP=100  THINKING_BUDGET=8000
ANTHROPIC_API_KEY loaded (length=108, prefix=sk-ant…)
Claude client ready (claude-opus-4-5, extended thinking budget=8000).  PIDP → ../data/15_llm_cluster_LA_claude/claude_pidp_assignments.csv


In [4]:
# ── Load & prepare ─────────────────────────────────────────────────────────────────────────────────
df_nl, df_merged, avail_profile_cols = _lcp.load_and_prepare(
    NL_PROFILE, LA_COUNTS, hier_col, WAVE, TEST_MODE, TEST_LA_CODES
)
all_groups = _lcp.build_groups(df_merged, hier_col)
la_k_map   = _lcp.compute_cluster_allocations(df_merged, hier_col, N_CLUSTERS_LOCAL, _lp)
to_run, n_done, n_total, n_remain = _lcp.build_resume_state(all_groups, API_OUT, hier_col)
leading      = ["ladcd", "ladnm"] + (["group"] if hier_col else [])
SUMMARY_COLS = _lcp.make_summary_cols(leading)

# ── Run LLM per LA × group ──────────────────────────────────────────────────────────────────────────
df_features  = df_nl.drop(columns=["nl_profile"], errors="ignore")
failed_pairs = []

print(f"\nStarting LLM calls for {len(to_run)} LA×group(s) …", flush=True)

for i, (_, meta) in enumerate(to_run.iterrows(), start=1):
    ladcd_val = meta["ladcd"]
    ladnm_val = meta["ladnm"]
    group_val = meta.get("group")
    emp_label = resolve_emp_label(group_val) if group_val is not None else None
    context   = f"{ladnm_val} — {emp_label or group_val or 'all respondents'}"

    top, deduped = _lcp.slice_and_rank(df_merged, ladcd_val, group_val, MAX_PIDPS_PER_GROUP)
    k_target = (
        la_k_map.get((ladcd_val, group_val), N_CLUSTERS_LOCAL)
        if hier_col and group_val is not None
        else N_CLUSTERS_LOCAL
    )
    k = min(k_target, len(deduped))

    print(
        f"\n[{n_done + i}/{n_total}, {n_remain - i} left] "
        f"[{ladnm_val} / {emp_label or group_val or '—'}]  "
        f"{len(top)} pidps → {len(deduped)} unique profiles → {k} clusters …",
        flush=True,
    )

    try:
        result = _lcp.call_llm_with_retry(
            _llm_chat,
            list(zip(deduped["compact_profile"], deduped["n"].astype(int))),
            k, context, group_val, RETRY_DELAY,
        )
        la_df_pidp = _lcp.build_pidp_df(result, deduped, top, ladcd_val, ladnm_val, group_val)
        _lcp.save_batch(
            la_df_pidp, result, df_features, _cs,
            PIDP_OUT, API_OUT, ladcd_val, ladnm_val, group_val,
            hier_col, WAVE, SUMMARY_COLS,
        )
    except Exception as exc:
        print(f"\n  ✗ FAILED [{ladnm_val} / {emp_label or group_val or '—'}]: {exc}", flush=True)
        traceback.print_exc()
        failed_pairs.append((ladcd_val, ladnm_val, group_val))

# ── Final summary ─────────────────────────────────────────────────────────────────────────────────
_lcp.print_final_summary(API_OUT, PIDP_OUT, failed_pairs, leading, resolve_emp_label)



Loading NL profiles …
  27,330 respondents loaded.
  Profile columns available: 9/9
Loading LA counts …
  7,969,122 rows, 346 LAs loaded.
  TEST MODE — 6 LA(s): ['Blackpool', 'County Durham', 'Hounslow', 'Islington', 'Newham', 'Tower Hamlets']
  Merged: 142,289 rows
  Building compact profile encodings …
  Unique compact profiles: 22,194

36 LA×group combination(s) to process:
    ladcd         ladnm  group
E06000009     Blackpool    5.0
E06000047 County Durham    5.0
E09000018      Hounslow    5.0
E09000019     Islington    5.0
E09000025        Newham    5.0
E09000030 Tower Hamlets    5.0
E06000009     Blackpool    1.0
E06000047 County Durham    1.0
E09000018      Hounslow    1.0
E09000019     Islington    1.0
E09000025        Newham    1.0
E09000030 Tower Hamlets    1.0
E06000009     Blackpool    8.0
E06000047 County Durham    8.0
E09000018      Hounslow    8.0
E09000019     Islington    8.0
E09000025        Newham    8.0
E09000030 Tower Hamlets    8.0
E06000047 County Durham    7.0